# Magnetic Reconnection: MHD Theory and Modelling — Implementation
## Paper #77: Pontin & Priest (2022), *Living Reviews in Solar Physics*

---

이 노트북은 Pontin & Priest (2022)의 주요 MHD 재결합 결과를 Python으로 재현한다.

This notebook reproduces the key MHD reconnection results of Pontin & Priest (2022) in Python.

**Sections / 섹션:**
1. Sweet-Parker vs Petschek 재결합률 비교 / rate comparison
2. Plasmoid instability 임계값 / threshold and growth rates
3. 2D X-type null point 장선 구조 / field-line topology
4. 3D spine-fan null point 시각화 / visualisation
5. 단순 2D 쌍극자에서의 QSL Q-factor / QSL in a simple 2D bipole
6. Lundquist number scaling and diffusion time / 런드퀴스트 수 스케일링

**Environment / 환경**: `study-with-ai` conda environment

**References / 참고**: Pontin & Priest (2022), Eqs. (1)-(2), (63)-(72), Sect. 2.1, 2.2, 2.6, 8.3

In [ ]:
# Standard imports for MHD reconnection calculations.
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.integrate import odeint

plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 12,
    'lines.linewidth': 1.6,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Libraries loaded.')

---

## 1. Sweet-Parker vs Petschek Reconnection Rate / 재결합률 비교

Sweet-Parker: $M_A = 1/\sqrt{S}$ (Pontin & Priest Eq. 66).

Petschek: $M_e^* \approx \pi/(8 \ln S)$ (Eq. 72).

코로나 $S \sim 10^{12}$–$10^{14}$ 영역에서 두 비율의 차이를 그린다.

We plot the two rates over the coronal regime $S \sim 10^{12}$–$10^{14}$.

In [ ]:
def sweet_parker_rate(S: np.ndarray) -> np.ndarray:
    """Compute normalised Sweet-Parker reconnection rate.

    Args:
        S: Lundquist number(s).

    Returns:
        Normalised reconnection rate M_A = 1/sqrt(S).
    """
    return 1.0 / np.sqrt(S)


def petschek_rate(S: np.ndarray) -> np.ndarray:
    """Compute maximum Petschek reconnection rate.

    Args:
        S: Lundquist (magnetic Reynolds) number(s).

    Returns:
        Normalised rate M_e* ~ pi/(8 ln S).
    """
    return np.pi / (8.0 * np.log(S))


# Scan Lundquist number over many decades.
S_array = np.logspace(3, 15, 200)
M_sp = sweet_parker_rate(S_array)
M_pe = petschek_rate(S_array)

fig, ax = plt.subplots()
ax.loglog(S_array, M_sp, label='Sweet-Parker  $1/\\sqrt{S}$', color='steelblue')
ax.loglog(S_array, M_pe, label='Petschek  $\\pi/(8\\ln S)$', color='crimson')
ax.axvline(1e12, color='grey', linestyle='--', alpha=0.6, label='Coronal $S\\sim 10^{12}$')
ax.axhline(0.01, color='green', linestyle=':', alpha=0.6, label='Universal rate $\\sim 0.01\\,v_A$')
ax.set_xlabel('Lundquist number $S$')
ax.set_ylabel('Normalised reconnection rate $M_A$')
ax.set_title('Sweet-Parker vs Petschek reconnection rates')
ax.legend()
plt.tight_layout()
plt.show()

S_coronal = 1e12
print(f'At coronal S = {S_coronal:.0e}:')
print(f'  Sweet-Parker rate = {sweet_parker_rate(S_coronal):.2e} v_A')
print(f'  Petschek rate     = {petschek_rate(S_coronal):.3f} v_A')
print(f'  Ratio (Petschek / SP) = {petschek_rate(S_coronal)/sweet_parker_rate(S_coronal):.1e}')

### Reconnection time-scales / 재결합 시간 척도

플레어 관찰 시간 척도(~100초)와 비교.

Compare with flare observational time-scale (~100 s).

In [ ]:
# Typical active-region parameters.
L = 1e7          # metres, typical active-region length
v_A = 1e6        # m/s, coronal Alfven speed
S_coronal = 1e12

tau_A = L / v_A                                 # Alfven time
tau_SP = L / (sweet_parker_rate(S_coronal) * v_A)  # Sweet-Parker time
tau_P = L / (petschek_rate(S_coronal) * v_A)    # Petschek time

print(f'Alfven crossing time  tau_A  = {tau_A:.1f} s')
print(f'Sweet-Parker time     tau_SP = {tau_SP:.2e} s  ({tau_SP / 3.15e7:.2f} years)')
print(f'Petschek time         tau_P  = {tau_P:.0f} s  ({tau_P / 60:.1f} min)')
print(f'Observed flare time   ~ 100-1000 s  -> Petschek matches, SP fails by ~1e5')

---

## 2. Plasmoid Instability Threshold / 플라즈모이드 불안정성 임계값

Loureiro et al. (2007): Sweet-Parker 시트의 tearing은 $S > S_c \sim 10^4$에서 발동.
성장률: $\gamma_{\max}\tau_A \sim S^{1/4}$, wavenumber: $k_{\max}L \sim S^{3/8}$.

Tearing of SP sheets for $S > S_c \sim 10^4$, with $\gamma_{\max}\tau_A \sim S^{1/4}$ and $k_{\max}L \sim S^{3/8}$.

In [ ]:
def plasmoid_growth_rate(S: np.ndarray) -> np.ndarray:
    """Scaling of the maximum plasmoid instability growth rate.

    Args:
        S: Lundquist number(s).

    Returns:
        gamma_max * tau_A ~ S^{1/4}.
    """
    return S ** 0.25


def plasmoid_wavenumber(S: np.ndarray) -> np.ndarray:
    """Scaling of the fastest-growing wavenumber.

    Args:
        S: Lundquist number(s).

    Returns:
        k_max * L ~ S^{3/8}.
    """
    return S ** 0.375


S_range = np.logspace(3, 14, 100)
S_crit = 1e4

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.loglog(S_range, plasmoid_growth_rate(S_range), color='crimson')
ax1.axvline(S_crit, color='grey', linestyle='--', label=f'$S_c = {S_crit:.0e}$')
ax1.set_xlabel('Lundquist number $S$')
ax1.set_ylabel('$\\gamma_{\\max}\\tau_A$')
ax1.set_title('Plasmoid growth rate scaling')
ax1.legend()

ax2.loglog(S_range, plasmoid_wavenumber(S_range), color='steelblue')
ax2.axvline(S_crit, color='grey', linestyle='--', label=f'$S_c = {S_crit:.0e}$')
ax2.set_xlabel('Lundquist number $S$')
ax2.set_ylabel('$k_{\\max} L$')
ax2.set_title('Fastest-growing wavenumber')
ax2.legend()

plt.tight_layout()
plt.show()

# Coronal plasmoid cascade example.
S_c_field = 1e13
print(f'For coronal sheet with S = {S_c_field:.0e}:')
print(f'  gamma_max * tau_A = {plasmoid_growth_rate(S_c_field):.1f}')
print(f'  k_max * L = {plasmoid_wavenumber(S_c_field):.2e}')
print(f'  => ~ {plasmoid_wavenumber(S_c_field):.0f} plasmoids at onset')
print(f'  Effective nonlinear rate ~ 1/sqrt(S_c) = {1/np.sqrt(1e4):.3f} v_A')

---

## 3. 2D X-type Null Point: Field Lines / 2D X-타입 영점: 장선

Linear 2D null: $\mathbf{B} = (B_0/r_0)(y, \bar{\alpha}^2 x)$ (Pontin & Priest Eq. 3).

- $\bar{\alpha}^2 > 0$: X-type (hyperbolic)
- $\bar{\alpha}^2 < 0$: O-type (elliptic)

X-point은 외부 경계가 자유로우면 "local collapse"로 불안정. X-points are unstable to local collapse with free boundaries.

In [ ]:
def null_2d_field(x: np.ndarray, y: np.ndarray, alpha2: float = 1.0) -> tuple:
    """Evaluate linear 2D null-point magnetic field.

    Args:
        x: X-coordinate array.
        y: Y-coordinate array.
        alpha2: Shape parameter (>0 gives X-type, <0 gives O-type).

    Returns:
        Tuple (B_x, B_y) with same shape as inputs.
    """
    return y, alpha2 * x


# Grid.
nx = 60
xs = np.linspace(-1, 1, nx)
ys = np.linspace(-1, 1, nx)
X, Y = np.meshgrid(xs, ys)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# X-type.
Bx, By = null_2d_field(X, Y, alpha2=1.0)
axes[0].streamplot(X, Y, Bx, By, color=np.hypot(Bx, By), cmap='viridis', density=1.4)
axes[0].plot(0, 0, 'rx', markersize=14, markeredgewidth=3, label='Null')
axes[0].plot([-1, 1], [-1, 1], 'w--', alpha=0.7, label='Separatrix')
axes[0].plot([-1, 1], [1, -1], 'w--', alpha=0.7)
axes[0].set_xlabel('$x/r_0$')
axes[0].set_ylabel('$y/r_0$')
axes[0].set_title('X-type null  ($\\bar{\\alpha}^2 = +1$)')
axes[0].legend()
axes[0].set_aspect('equal')

# O-type.
Bx, By = null_2d_field(X, Y, alpha2=-1.0)
axes[1].streamplot(X, Y, Bx, By, color=np.hypot(Bx, By), cmap='viridis', density=1.4)
axes[1].plot(0, 0, 'ro', markersize=10, label='Null (O-type)')
axes[1].set_xlabel('$x/r_0$')
axes[1].set_ylabel('$y/r_0$')
axes[1].set_title('O-type null  ($\\bar{\\alpha}^2 = -1$)')
axes[1].legend()
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

---

## 4. 3D Spine-Fan Null Point / 3D 스파인-팬 영점

Simplest 3D null (Pontin & Priest Eq. 5):
$$\mathbf{B} = (x, y, -2z)$$

- **Spine**: z-axis (the single field line reaching the null from above/below)
- **Fan surface**: $z = 0$ (xy-plane; family of field lines spreading from null)

Field lines satisfy $y = Cx$, $z = K/x^2$.

In [ ]:
def null_3d_field(x: np.ndarray, y: np.ndarray, z: np.ndarray) -> tuple:
    """Evaluate proper radial 3D null (Pontin-Priest Eq. 5).

    Args:
        x: X-coordinate(s).
        y: Y-coordinate(s).
        z: Z-coordinate(s).

    Returns:
        Tuple (B_x, B_y, B_z).
    """
    return x, y, -2.0 * z


def trace_3d(seed: np.ndarray, s_range: np.ndarray) -> np.ndarray:
    """Trace a 3D field line from a seed point.

    Args:
        seed: Starting point [x0, y0, z0].
        s_range: Parameter values for integration.

    Returns:
        Array of shape (len(s_range), 3) of traced coordinates.
    """
    def rhs(r, _s):
        bx, by, bz = null_3d_field(r[0], r[1], r[2])
        norm = max(np.sqrt(bx * bx + by * by + bz * bz), 1e-12)
        return [bx / norm, by / norm, bz / norm]

    return odeint(rhs, seed, s_range)


fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot fan surface field lines (starting in xy-plane).
s_pos = np.linspace(0, 3, 150)
s_neg = np.linspace(0, -3, 150)
for angle in np.linspace(0, 2 * np.pi, 10, endpoint=False):
    seed = np.array([0.15 * np.cos(angle), 0.15 * np.sin(angle), 0.001])
    line_out = trace_3d(seed, s_pos)
    line_in = trace_3d(seed, s_neg)
    ax.plot(line_out[:, 0], line_out[:, 1], line_out[:, 2], color='steelblue', alpha=0.8, linewidth=1.2)
    ax.plot(line_in[:, 0], line_in[:, 1], line_in[:, 2], color='steelblue', alpha=0.8, linewidth=1.2)

# Plot spine (z-axis).
z_spine = np.linspace(-1.5, 1.5, 50)
ax.plot(np.zeros_like(z_spine), np.zeros_like(z_spine), z_spine, color='red', linewidth=3, label='Spine')

# Mark null.
ax.scatter([0], [0], [0], color='black', s=80, label='Null point')

# Draw fan plane (faint).
xp, yp = np.meshgrid(np.linspace(-1, 1, 10), np.linspace(-1, 1, 10))
ax.plot_surface(xp, yp, np.zeros_like(xp), alpha=0.12, color='yellow')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('3D proper radial null: spine (red) + fan (plane)')
ax.legend()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1.5, 1.5)
plt.tight_layout()
plt.show()

---

## 5. QSL and Squashing Factor Q in a 2D Sheared Null / 2D shear null에서 QSL과 Q 인자

Pontin & Priest Sect. 2.6.1의 예시: $\mathbf{B} = (x, -y, l)$를 박스 $[-1/2, 1/2]^2 \times [0, 1]$에서. $l \ll 1$일 때 평면 $x = 0$과 $y = 0$이 QSL이 된다. Mapping $z=0 \to z=1$: $x_1 = x_0 e^{z_1/l}$, $y_1 = y_0 e^{-z_1/l}$.

We compute the squashing factor $Q$ numerically and show a large $Q$ band identifying the QSL.

In [ ]:
def q_factor_sheared_null(x0: np.ndarray, y0: np.ndarray, l: float = 0.1) -> np.ndarray:
    """Compute squashing factor Q for the Priest-Demoulin sheared-null example.

    Mapping from z=0 to z=1 is:
        x1 = x0 * exp( 1/l ),
        y1 = y0 * exp(-1/l ).

    The mapping is uniform so Q is spatially constant in this simple model;
    we evaluate it analytically and also on a grid to illustrate the code path.

    Args:
        x0: Footpoint X-coordinates.
        y0: Footpoint Y-coordinates.
        l: Guide-field strength parameter (smaller = thinner QSL).

    Returns:
        Squashing factor Q over the input grid.
    """
    a = np.exp(1.0 / l)
    b = np.exp(-1.0 / l)

    # Jacobian of footpoint mapping
    # dX/dx = a, dX/dy = 0, dY/dx = 0, dY/dy = b
    N_sq = a * a + b * b

    # B_n ratio = 1 for this configuration (uniform guide field)
    Q = N_sq / 1.0

    # Broadcast to grid shape.
    return np.broadcast_to(Q, x0.shape).copy()


# Now a more realistic case: shear increases toward null.
def q_factor_localised(x0: np.ndarray, y0: np.ndarray, width: float = 0.05) -> np.ndarray:
    """Compute a spatially varying squashing factor for a 2D bipole-like field.

    Uses a toy model where field lines near the polarity inversion line (x=0)
    experience much larger mapping gradients than those far away. This mimics
    the QSL structure seen in the Titov-Demoulin flux-rope model.

    Args:
        x0: Footpoint X-coordinates.
        y0: Footpoint Y-coordinates.
        width: Width of the QSL band (smaller = thinner).

    Returns:
        Squashing factor Q over the grid.
    """
    # Mapping gradient grows exponentially near x0 = 0 (polarity inversion line).
    stretch = np.exp(1.0 / (width + np.abs(x0)))
    compress = np.exp(-1.0 / (width + np.abs(x0)))
    N_sq = stretch ** 2 + compress ** 2
    return N_sq


# Visualise.
nxy = 200
xv = np.linspace(-1, 1, nxy)
yv = np.linspace(-1, 1, nxy)
X0, Y0 = np.meshgrid(xv, yv)
Q_grid = q_factor_localised(X0, Y0, width=0.08)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(np.log10(Q_grid), extent=(-1, 1, -1, 1), origin='lower', cmap='hot')
cb = plt.colorbar(im, ax=ax)
cb.set_label('$\\log_{10} Q$')
ax.axvline(0, color='cyan', linestyle='--', label='Polarity inversion / QSL')
ax.set_xlabel('$x_0$ footpoint')
ax.set_ylabel('$y_0$ footpoint')
ax.set_title('Squashing factor $Q$ (toy bipole model)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Q in QSL band ( x ~ 0 ): ~ {Q_grid[:, nxy // 2].max():.1e}  (QSL if Q >> 2)')
print(f'Q far from QSL ( x = 1): ~ {Q_grid[:, -1].max():.2f}  (not a QSL)')

---

## 6. Lundquist Number Scaling: Diffusion Times / 런드퀴스트 수 스케일링

코로나의 확산 시간 스케일을 자기 reynolds 수 및 길이 스케일의 함수로 계산.

Compute magnetic diffusion time as a function of length-scale and $S$.

In [ ]:
def diffusion_time(L: float, eta: float) -> float:
    """Magnetic diffusion time tau_d = L^2 / eta (Eq. 63 adjacent text).

    Args:
        L: Length scale in metres.
        eta: Magnetic diffusivity in m^2 / s.

    Returns:
        Diffusion time in seconds.
    """
    return L ** 2 / eta


def spitzer_eta(T_K: float) -> float:
    """Very rough Spitzer magnetic diffusivity (SI).

    Args:
        T_K: Temperature in Kelvin.

    Returns:
        Magnetic diffusivity in m^2 / s, scaling as T^{-3/2}.
    """
    return 1e9 * T_K ** (-1.5)


# Scan.
T_corona = 1e6       # K
T_chromo = 1e4       # K
eta_cor = spitzer_eta(T_corona)
eta_chr = spitzer_eta(T_chromo)
print(f'Coronal    eta (T=1e6 K): {eta_cor:.2e} m^2/s')
print(f'Chromosph  eta (T=1e4 K): {eta_chr:.2e} m^2/s')

L_array = np.logspace(2, 8, 200)  # 100 m to 100 Mm
tau_d_cor = np.array([diffusion_time(L_, eta_cor) for L_ in L_array])
tau_d_chr = np.array([diffusion_time(L_, eta_chr) for L_ in L_array])

fig, ax = plt.subplots()
ax.loglog(L_array / 1e6, tau_d_cor, label=f'Corona (T=1e6 K)', color='orange')
ax.loglog(L_array / 1e6, tau_d_chr, label=f'Chromosphere (T=1e4 K)', color='navy')
ax.axhline(100, color='grey', linestyle='--', label='Flare 100 s')
ax.axhline(86400, color='grey', linestyle=':', label='1 day')
ax.set_xlabel('Length scale L  [Mm]')
ax.set_ylabel('Diffusion time $\\tau_d = L^2/\\eta$  [s]')
ax.set_title('Pure diffusion time-scale vs length scale')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nAt L = 10 Mm coronal length, tau_d = {diffusion_time(1e7, eta_cor):.2e} s')
print('  -> far longer than observed flare time: need Petschek or plasmoid regime')

---

## Summary / 요약

이 노트북에서 우리는 Pontin & Priest (2022)의 핵심 양적 결과를 재현했다:

In this notebook we reproduced the key quantitative results of Pontin & Priest (2022):

1. **Sweet-Parker ($1/\sqrt{S}$) vs Petschek ($\pi/(8\ln S)$)**: 코로나 $S\sim 10^{12}$에서 Petschek이 SP보다 약 $10^5$배 빠름. Petschek is $\sim 10^5\times$ faster than SP at coronal $S$.
2. **Plasmoid instability**: $S_c \sim 10^4$ threshold, $\gamma\tau_A \sim S^{1/4}$, $kL \sim S^{3/8}$ 스케일링. Effective rate $\sim 0.01\,v_A$—$S$와 거의 무관. Nonlinear rate is $\sim 0.01\,v_A$, nearly $S$-independent.
3. **2D null points**: X-type (hyperbolic separatrices) vs O-type (circular, flux creation/destruction).
4. **3D null structure**: spine curve + fan surface, fundamentally different from 2D.
5. **QSL identification** through squashing factor $Q \gg 2$; observed QSLs match flare ribbons.
6. **Diffusion time-scales** in corona vs chromosphere: 순수 확산만으로는 플레어 설명 불가. Pure diffusion cannot explain flares.

**다음 단계 / Next steps**: full resistive MHD simulations, 3D plasmoid cascade, spine-fan null-point collapse simulation, Titov-Démoulin flux rope QSL mapping with real active-region magnetograms.